# Wolvercote format — end-to-end demo

This notebook demonstrates the Wolvercote format for describing bacterial genome organisation.
It covers:

1. **Basic usage** — parsing, validation, round-trip serialisation
2. **Rendering** — SVG and publication-quality PNG
3. **NCBI data** — download a real GenBank record and convert it
4. **The Russian dolls figure** — visualising the same mobile element in multiple hosts

Install the package first:
```bash
pip install wolvercote            # core
pip install wolvercote[biopython] # + GenBank support
```
Or from source (this repo):
```bash
pip install -e python/
```

In [ ]:
import sys, pathlib
# If running from the repo root without installing:
repo = pathlib.Path.cwd().parent
if (repo / 'python').exists():
    sys.path.insert(0, str(repo / 'python'))

from wolvercote import parse, validate, is_valid, render_svg, to_wolvercote
from wolvercote.render_png import render_png
print('wolvercote ready')

## 1. Parsing and validation

The Wolvercote format grammar at a glance:

| Construct | Meaning |
|-----------|--------|
| `()label` | Chromosome named *label* |
| `{}label` | Plasmid / MGE named *label* |
| `({}Tn3)chr` | Chromosome with Tn3 on its border |
| `{ {}blaKPC }pKpQIL` | Plasmid containing blaKPC |
| `A , B` | Two replicons in the same cell |
| `A ; B` | Two separate cells |
| `[k="v"]` | Attribute annotation |

In [ ]:
# Simple parse
cs = parse('()chr1, {}pBAD')
print('Cells:', len(cs.cells))
print('Replicons in cell 1:', [(r.__class__.__name__, r.label) for r in cs.cells[0].replicons])

In [ ]:
# Validation
print(validate('()unclosed'))   # error message
print(is_valid('()chr1'))       # True
print(is_valid('(unclosed'))    # False

In [ ]:
# Round-trip: parse → serialise back
original = '({}ISKpn6, {}ISKpn7)chromosome, { { {}blaKPC-2 }Tn4401 }pKpQIL'
cs = parse(original)
print('Round-trip:', to_wolvercote(cs))

## 2. Rendering

### SVG

In [ ]:
from IPython.display import SVG, display

cs = parse('({}integron1)chromosome, { {}Tn3, {}IS26 }pR6K')
svg = render_svg(cs)
display(SVG(data=svg))

### Publication-quality PNG (matplotlib)

In [ ]:
from IPython.display import Image, display
import io

cs = parse('({}ISKpn6, {}ISKpn7)chromosome, { { {}blaKPC-2 }Tn4401, {}ISKpn5 }pKpQIL, {}pColE1')
png_bytes = render_png(cs, dpi=150, legend=True)
display(Image(data=png_bytes))

## 3. Real-world data from NCBI

We download the KPC-bearing plasmid pKpQIL (GQ347594.1) from Klebsiella pneumoniae,
a key element in carbapenem-resistance transmission (Leavitt et al. 2010).
This plasmid carries a Tn4401 transposon which in turn carries *bla*KPC-2.

> **Data source**: NCBI GenBank GQ347594.1 (pKpQIL plasmid, *K. pneumoniae* 4/KPC)

In [ ]:
# Download GenBank record from NCBI using Biopython Entrez
from Bio import Entrez, SeqIO
import tempfile, pathlib

Entrez.email = 'demo@example.com'  # required by NCBI

accessions = [
    'GQ347594',  # pKpQIL plasmid (KPC + Tn4401)
    'CP000647',  # K. pneumoniae MGH 78578 chromosome
]

gbk_path = pathlib.Path('/tmp/kpn_demo.gbk')
with Entrez.efetch(db='nucleotide', id=','.join(accessions),
                   rettype='gbwithparts', retmode='text') as handle:
    gbk_path.write_text(handle.read())

print('Downloaded', gbk_path, '—', gbk_path.stat().st_size // 1024, 'KB')

In [ ]:
from wolvercote.genbank import from_genbank_file

cs = from_genbank_file(gbk_path)
wt = to_wolvercote(cs)
print('Wolvercote string:')
print(wt)
print()
for cell in cs.cells:
    for r in cell.replicons:
        kind = 'Chromosome' if r.__class__.__name__ == 'ChromosomeNode' else 'MGE/Plasmid'
        print(f'  {kind}: {r.label} ({r.size_bp:,} bp)' if r.size_bp else f'  {kind}: {r.label}')
        for ch in r.children:
            print(f'    └─ {ch.label}')

In [ ]:
png_bytes = render_png(cs, dpi=150, legend=True)
display(Image(data=png_bytes))

## 4. The Russian dolls figure

A key use case for Wolvercote is showing how the same mobile element appears in multiple
bacterial hosts — the "Russian dolls" nested containment structure.

Here we show pKpQIL (carrying Tn4401 → blaKPC-2) present in both:
- *K. pneumoniae* (cell 1): with additional IS elements on the chromosome
- *E. coli* transconjugant (cell 2): acquired pKpQIL by conjugal transfer

The shared plasmid highlights inter-species spread of carbapenem resistance.

In [ ]:
russian_dolls = (
    '({}ISKpn6, {}ISKpn7)chromosome, { { {}blaKPC-2 }Tn4401, {}ISKpn5 }pKpQIL, {}pColE1'
    ' ; '
    '()chromosome, { { {}blaKPC-2 }Tn4401 }pKpQIL'
)

cs = parse(russian_dolls)
png_bytes = render_png(cs, dpi=200, legend=True)
display(Image(data=png_bytes))

# Save for the paper
pathlib.Path('russian_dolls.png').write_bytes(png_bytes)
print('Saved russian_dolls.png')

## Format grammar reference

```
cell_set   := cell ( ';' cell )*
cell       := replicon ( ',' replicon )*
replicon   := chromosome | mge
chromosome := '(' mge_list ')' label attrs?
mge        := '{' mge_list '}' label attrs?
mge_list   := ( mge ( ',' mge )* )?
label      := [a-zA-Z0-9_\-./ ]*
attrs      := '[' attr ( ',' attr )* ']'
attr       := key '=' '"' value '"'
```

**Examples**

| Format string | Meaning |
|---|---|
| `()chr1` | Single unnamed chromosome |
| `()chr1, {}pBAD` | Chromosome + one plasmid |
| `({}Tn3)chr1` | Tn3 integrated in chromosome |
| `{ {}blaKPC-2 }pKpQIL` | Plasmid containing a resistance gene |
| `{ { {}blaKPC-2 }Tn4401 }pKpQIL` | blaKPC-2 inside Tn4401 inside pKpQIL |
| `()A ; ()B` | Two different bacterial cells |
| `()chr[organism="E. coli"]` | Chromosome with organism attribute |